In [ ]:
import nltk
from nltk.corpus.reader import PlaintextCorpusReader
corpus = PlaintextCorpusReader(root="data", fileids=["toy_corpus.txt"])
token = corpus.words()
tokens = [word.lower() for word in corpus.words()]
symbols=['.',',',':',';','!','?','-','\'','\"','(',')','[',']']
translation_table = str.maketrans('', '', ''.join(symbols))
token_without_symbols = [token.translate(translation_table) for token in tokens]
token_without_symbols_empty = [token for token in token_without_symbols if token != ""]
words=token_without_symbols_empty



In [ ]:
words_map={}
for word in words:
    if(word in words_map):
        words_map[word]+=1
    else:
        words_map[word]=1
sorted_by_value_then_key = dict(sorted(words_map.items(), key=lambda item: (-item[1], item[0])))
top_5 = list(sorted_by_value_then_key.items())[:5]
top_5_dict = dict(top_5)
print("\n".join(list(top_5_dict.keys())))

In [ ]:
set(words)
print("\n".join(words))

In [ ]:
computer_index=words.index("computer")
index=[0,computer_index,(len(words))-2]
def window(index):
    for i in index:
        window_table = [i-3, i-2,i-1,i+1,i+2,i+3]
        window_table=[ value for value in window_table if value>=0 and value<len(words)]
        window_table=[ words[value] for value in window_table]
        print(",".join(window_table))

window(index)


In [ ]:
words = token_without_symbols_empty

index_languages = [index for index, value in enumerate(words) if value == 'languages']
index_linguists = [index for index, value in enumerate(words) if value == 'linguists']

def get_context_window(index_list):
    context_map = {}
    for idx in index_list:
        window_indices = [i for i in range(idx - 3, idx + 4) if i >= 0 and i < len(words) and i != idx]
        window_words = [words[i] for i in window_indices]
        for word in window_words:
            context_map[word] = context_map.get(word, 0) + 1
    return context_map

languages_map = get_context_window(index_languages)
linguists_map = get_context_window(index_linguists)

languages_map_str = ', '.join(f"{k}: {v}" for k, v in languages_map.items())
print(languages_map_str)
linguists_map_str = ', '.join(f"{k}: {v}" for k, v in linguists_map.items())
print(linguists_map_str)

dot_product = sum(languages_map.get(word, 0) * linguists_map.get(word, 0) for word in set(languages_map) | set(linguists_map))
norm_languages = sum(value ** 2 for value in languages_map.values()) ** 0.5
norm_linguists = sum(value ** 2 for value in linguists_map.values()) ** 0.5
cosine_similarity = dot_product / (norm_languages * norm_linguists) if norm_languages and norm_linguists else 0

print(cosine_similarity)


In [ ]:
import nltk
from nltk.corpus.reader.plaintext import PlaintextCorpusReader
import re
from collections import Counter
import zipfile
corpus_root = 'corpora/'

with zipfile.ZipFile('corpora/corpus.zip', 'r') as zip_ref:
    zip_ref.extract('corpus.txt', path=corpus_root)

corpus = PlaintextCorpusReader(corpus_root, 'corpus.txt')

tokens = [word.lower() for word in corpus.words()] 
clean_tokens = [re.sub(r"[-.,:;!?\'\"()\[\]]", "", word) for word in tokens] 
clean_tokens = [word for word in clean_tokens if word]
token_counts = Counter(clean_tokens)
vocab = {word for word, count in token_counts.items() if count >= 5}
final_tokens = [word if word in vocab else "<UNK>" for word in clean_tokens]


from collections import Counter, defaultdict
from scipy.sparse import csr_matrix
import numpy as np

window_size = 3 
epsilon = 2e-5

vocab_list = list(vocab)
vocab_index = {word: i for i, word in enumerate(vocab_list)} 
cooccurrence_counts = defaultdict(Counter)

for i, word in enumerate(final_tokens):  
    if word in vocab_index:
        target_idx = vocab_index[word]
        for j in range(max(0, i - window_size), min(len(final_tokens), i + window_size + 1)):
            if i != j and final_tokens[j] in vocab_index:
                context_idx = vocab_index[final_tokens[j]]
                cooccurrence_counts[target_idx][context_idx] += 1
total_cooccurrences = sum(sum(context.values()) for context in cooccurrence_counts.values())
word_counts = np.zeros(len(vocab_list))
context_counts = np.zeros(len(vocab_list))

for word_idx, contexts in cooccurrence_counts.items():
    word_counts[word_idx] = sum(contexts.values())
    for context_idx, count in contexts.items():
        context_counts[context_idx] += count

data, rows, cols = [], [], []
for word_idx, contexts in cooccurrence_counts.items():
    for context_idx, count in contexts.items():
        p_wc = count / total_cooccurrences
        p_w = word_counts[word_idx] / total_cooccurrences
        p_c = context_counts[context_idx] / total_cooccurrences
        ppmi = max(0, np.log2((p_wc + epsilon) / (p_w * p_c)))
        
        if ppmi > 0:
            rows.append(word_idx)
            cols.append(context_idx)
            data.append(ppmi)

ppmi_matrix = csr_matrix((data, (rows, cols)), shape=(len(vocab_list), len(vocab_list)))
from sklearn.metrics.pairwise import cosine_similarity

city_idx = vocab_index.get("army")
if city_idx is None:
    raise ValueError("The word 'city' is not in the vocabulary.")

city_vector = ppmi_matrix[city_idx]

similarities = []
for word, idx in vocab_index.items():
    if word != "army":
        word_vector = ppmi_matrix[idx]
        similarity = cosine_similarity(city_vector, word_vector)[0, 0]
        similarities.append((word, similarity))

closest_words = sorted(similarities, key=lambda x: x[1], reverse=True)[:5]
closest_words_set = {word for word, _ in closest_words}

print(closest_words_set)
#{'local', 'media', 'daily', 'newspaper', 'forks'} news
#{'located', 'a', 'county', 'in', 'location'} city
#{'navy', 'war', 'military', 'fort', 'camp'} army


In [ ]:
from gensim.models import Word2Vec
import nltk
import re
from nltk.corpus import PlaintextCorpusReader
import zipfile
corpus_root = 'corpora/'

with zipfile.ZipFile('corpora/corpus.zip', 'r') as zip_ref:
    zip_ref.extract('corpus.txt', path=corpus_root)
corpus = PlaintextCorpusReader(corpus_root, 'corpus.txt')
raw_text = corpus.raw()
raw_sentences = nltk.sent_tokenize(raw_text)

sentences = []
for sentence in raw_sentences:
    tokens = nltk.word_tokenize(sentence)
    clean_tokens = [re.sub(r"[-.,:;!?\'\"()\[\]]", "", word).lower() for word in tokens]
    clean_tokens = [token for token in clean_tokens if token]
    sentences.append(clean_tokens)


embedding_dim = 150
window_size = 7
skipgram = 1 
negative_samples = 4
epochs = 40
workers = 1  
model = Word2Vec(
    sentences=sentences,
    vector_size=embedding_dim, 
    window=window_size,
    sg=skipgram,
    negative=negative_samples,
    epochs=epochs,
    workers=workers
)

closest_words = model.wv.most_similar("news", topn=5)
closest_words_set = {word for word, _ in closest_words}
print(closest_words_set)
#{'borough', 'village', 'cdp', 'cities', 'town'} city
#{'corps', 'marched', 'regiment', 'military', 'encamped'} army
#{'daily', 'media', 'broadcast', 'tribune', 'newspaper'} news


#{'municipality', 'citys', 'town', 'municipal', 'mayor'}


In [ ]:
from nltk.corpus.reader.plaintext import PlaintextCorpusReader
import re
from collections import Counter

import zipfile
corpus_root = 'corpora/'

with zipfile.ZipFile('corpora/corpus.zip', 'r') as zip_ref:
    zip_ref.extract('corpus.txt', path=corpus_root)
corpus = PlaintextCorpusReader(corpus_root, 'corpus.txt')

tokens = [word.lower() for word in corpus.words()]  
cleaned_tokens = [re.sub(r"[-.,:;!?\'\"()\[\]]", "", token) for token in tokens]
cleaned_tokens = [token for token in cleaned_tokens if token]

token_counts = Counter(cleaned_tokens)
vocabulary = {token for token, count in token_counts.items() if count >= 5}

processed_corpus = [
    token if token in vocabulary else "<UNK>" for token in cleaned_tokens
]
from collections import defaultdict
from scipy.sparse import coo_matrix
import numpy as np

window_size = 3

cooccurrence = defaultdict(lambda: defaultdict(int))

for i, token in enumerate(processed_corpus):
    context_start = max(i - window_size, 0)
    context_end = min(i + window_size + 1, len(processed_corpus))
    for j in range(context_start, context_end):
        if i != j:
            cooccurrence[token][processed_corpus[j]] += 1

vocab_list = sorted(vocabulary | {"<UNK>"})
vocab_index = {word: idx for idx, word in enumerate(vocab_list)}
rows, cols, data = [], [], []

for word, contexts in cooccurrence.items():
    for context_word, count in contexts.items():
        rows.append(vocab_index[word])
        cols.append(vocab_index[context_word])
        data.append(count)

coo_matrix = coo_matrix((data, (rows, cols)), shape=(len(vocab_list), len(vocab_list)))
from sklearn.feature_extraction.text import TfidfTransformer

tfidf_transformer = TfidfTransformer()
tfidf_matrix = tfidf_transformer.fit_transform(coo_matrix)
from sklearn.metrics.pairwise import cosine_similarity

def compute_cosine_similarity(word1, word2, tfidf_matrix, vocab_index):
    idx1 = vocab_index.get(word1, vocab_index["<UNK>"])
    idx2 = vocab_index.get(word2, vocab_index["<UNK>"])
    vec1 = tfidf_matrix[idx1]
    vec2 = tfidf_matrix[idx2]
    return cosine_similarity(vec1, vec2)[0][0]
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def find_closest_words(target_word, tfidf_matrix, vocab_index, top_n=5):
    if target_word not in vocab_index:
        raise ValueError()

    target_idx = vocab_index[target_word]
    target_vector = tfidf_matrix[target_idx]

    similarities = cosine_similarity(tfidf_matrix, target_vector).flatten()

    similar_indices = np.argsort(-similarities) 
    closest_words = [
        vocab_list[idx] for idx in similar_indices if vocab_list[idx] != target_word
    ][:top_n]

    return set(closest_words)

closest_to_city = find_closest_words("city", tfidf_matrix, vocab_index, top_n=5)
print(closest_to_city)
#{'being', 'only', 'by', '<UNK>', 'newspaper'} news
#{'house', 'last', 'by', 'following', 'band'} army
#{'borough', 'community', 'village', 'cdp', 'town'} city

{'borough', 'community', 'village', 'cdp', 'town'}


In [ ]:
from gensim.models import Word2Vec
import nltk
import re
from nltk.corpus import PlaintextCorpusReader
import zipfile
from gensim.models import Word2Vec
from gensim.models import KeyedVectors
import gensim.downloader

corpus_root = 'corpora/'

model = gensim.downloader.load('word2vec-google-news-300')

raw_text = model.raw()
raw_sentences = nltk.sent_tokenize(raw_text)

sentences = []
for sentence in raw_sentences:
    tokens = nltk.word_tokenize(sentence)
    clean_tokens = [re.sub(r"[-.,:;!?\'\"()\[\]]", "", word).lower() for word in tokens]
    clean_tokens = [token for token in clean_tokens if token]
    sentences.append(clean_tokens)


embedding_dim = 150
window_size = 7
skipgram = 1 
negative_samples = 4
epochs = 40
workers = 1  
model = Word2Vec(
    sentences=sentences,
    vector_size=embedding_dim, 
    window=window_size,
    sg=skipgram,
    negative=negative_samples,
    epochs=epochs,
    workers=workers
)

closest_words = model.wv.most_similar("news", topn=5)
closest_words_set = {word for word, _ in closest_words}
print(closest_words_set)